# NB03: Data Analysis

The research question underpinning this project is: is winning a professional mens singles tennis match better explained by hitting more winners (an aggressive, risk taking style), or by comitting fewer unforced errors (a consistent, risk-averse style)?

Tennis is a very complex sport, which makes it difficult to draw concrete conclusions with merely correlational data and analysis. In a game where tactics, matchups and environmental conditions are so variable, raw numbers can only tell a part of the story and should not be treated as the definitive truth.



## Setup

In [13]:
import pandas as pd
import plotly.express as px

Each row in the prepared table represents one match, and has the variable `tournament_name`, alongside each player's winners, unforced errors, aces and double faults.

In [14]:
df = pd.read_csv("../data/processed/cleaned.csv")

df["Extra winners hit by the match winner"] = df["winner_winners"] - df["loser_winners"]
df["Fewer errors made by the match winner"] = -(df["winner_ue"] - df["loser_ue"])

df

,tournament_name,winner_winners,winner_ue,winner_aces,winner_df,loser_winners,loser_ue,loser_aces,loser_df,Extra winners hit by the match winner,Fewer errors made by the match winner
0,Australian Open,14,19,0,2,22,35,6,4,-8,16
1,Australian Open,16,7,4,0,14,27,3,3,2,20
2,Australian Open,37,28,6,2,21,31,1,4,16,3
3,Australian Open,20,20,2,1,9,29,1,3,11,9
4,Australian Open,8,13,1,1,11,29,2,2,-3,16
...,...,...,...,...,...,...,...,...,...,...,...
707,Wimbledon,36,33,7,0,72,59,19,7,-36,26
708,Wimbledon,43,47,14,6,74,61,29,5,-31,14
709,Wimbledon,27,15,8,1,21,41,6,2,6,26
710,Wimbledon,29,15,14,2,31,28,17,2,-2,13


With a scatter plot (each point = one match), we can identify a general pattern for how each individual game is distributed and identify if a relationship might exist between winners and unforced errors.

In [15]:
scatterplot = px.scatter(df, x="Extra winners hit by the match winner", y="Fewer errors made by the match winner",
    title="Extra winners come with extra errors, and vice versa",
    subtitle="Each point represents the error differential against the winner differential for one match",
    opacity=0.5,
    width=800,
    height=800,
)

scatterplot.show()

This scatter plot gives us an idea of what the data points are. As we can see, the matches follow this negative correlation, where hitting more winners normally comes with a smaller error margin (or hitting more errrors than the opponent). This results in more results lying in this range around +20 points over their opponent on this metric (e.g 10 more winners + 10 less errors, or 30 more winners - 10 more errors etc).

We are trying to examine which one of these two is more conducive to winning matches. This means we want to investigate whether this trend is more packed towards the bottom right or if it is more packed towards the top left of the scatter plot.

To do that, we'll construct comparative box plots, which requires the dataframe to be converted into long form:

In [16]:
# stacks the two margins into one column "metric" so that plotly can plot them side by side
df_long = df.melt(
    id_vars=["tournament_name"],
    value_vars=["Extra winners hit by the match winner", "Fewer errors made by the match winner"],
    var_name="metric",
    value_name="margin"
)

df_long

,tournament_name,metric,margin
0,Australian Open,Extra winners hit by the match winner,-8
1,Australian Open,Extra winners hit by the match winner,2
2,Australian Open,Extra winners hit by the match winner,16
3,Australian Open,Extra winners hit by the match winner,11
4,Australian Open,Extra winners hit by the match winner,-3
...,...,...,...
1419,Wimbledon,Fewer errors made by the match winner,26
1420,Wimbledon,Fewer errors made by the match winner,14
1421,Wimbledon,Fewer errors made by the match winner,26
1422,Wimbledon,Fewer errors made by the match winner,13


In [17]:
fig1 = px.box(df_long, x="metric", y="margin", color="metric",
            title="Players who win typically have a higher margin of winners than unforced errors",
            subtitle="Side by side of how many more/less winners/unforced errors the winner of a match had against the loser",
            labels = {
                "metric": "Metric",
                "margin": "Differential",
            },
            width=800,
            height=800,
            )

fig1.update_layout(showlegend=False)

fig1.show()

Across all the matches, the median margin in winners hit by the match winner is +7, slightly higher than the error differential at +6 (6 less errors). Both distributions are extremely wide, with the error box having a slightly larger range but less outliers. The interquartile ranges are quite similar, implying that matches are not won just through one style, and that winning typically requires some combination of more winners and less errors (as supported by the scatterplot). 

The second figure splits by surface. Surface is a match variable that is very unique to tennis, and they behave as such (slowest to fastest):
1. Clay Court (French Open): made up of crushed brick/stone, slows down pace of play significantly and rewards patience, defense and consistency.
2. Hard Court (Australian Open): made up of concrete or asphalt, with a medium pace of play on which a balanced playstyle is strongest.
3. Grass Court (Wimbledon): made up of natural turf, with low bounces and fast play, favouring big servers and aggressive play.

This rhetoric is supported by the following chart. The Australian Open and Wimbledon share the same pattern from the complete dataset, with a bigger median difference in winners that in errors (+8 vs +7 and +7 vs +6 respectively), but the French Open is the only surface where the errors margin edges ahead of the median winners at +6 (6 less errors) and +5 (5 extra winners). 

Some more interesting patterns from the box-plots: The error box for the French Open has the widest range, which makes sense; the matches on clay last longer given the slower points extending rallies, which not only allows for more opportunities to make errors, but also introduces more fatigue. Wimbledon also has the more dense upper range on the winners box, which shows that aggressive play is more prevalent on grass.

In [18]:
fig2 = px.box(df_long, x="tournament_name", y="margin", color="metric",
            title="Clay (French Open) is the only surface where the median error differential is larger than the median winner differential",
            subtitle="Side by side of how many more/less winners/unforced errors the winner of a match had against the loser (by tournament)",
            labels = {
                "metric": "",
                "margin": "Margin",
                "tournament_name": "Tournament",
            },
            width=1200,
            height=800,
            )


fig2.update_layout(
    # move the legend from the side to the bottom
    legend=dict(
        orientation="h",  
        yanchor="top",
        y=-0.08, 
        xanchor="center",
        x=0.5
    )
)

fig2.show()

The data until this point has been using the raw value for winners and unforced errors. However, this is somewhat of a misleading statistic, as it includes aces (added to winners) and double faults (added to errors). While hitting more aces/double faults is a little bit indicative of aggression, it is moreso a measure of how well a player serves and how important it is to their game. 

So now, we create columns that have the aces and double faults removed, to see the risk tradeoff more clearly.

In [19]:
df["Extra winners hit by the match winner (in rally)"] = df["Extra winners hit by the match winner"] - (df["winner_aces"] - df["loser_aces"])
df["Fewer errors made by the match winner (in rally)"] = df["Fewer errors made by the match winner"] - (df["loser_df"] - df["winner_df"])

df_long_rally = df.melt(
    id_vars=["tournament_name"],
    value_vars=["Extra winners hit by the match winner (in rally)", "Fewer errors made by the match winner (in rally)"],
    var_name="metric",
    value_name="margin"
)

In [20]:
fig1_rally = px.box(df_long_rally, x="metric", y="margin", color="metric",
            title="Removing serves lowers the winners margin but leaves errors largely unchanged",
            subtitle="Side by side of how many more/less winners/unforced errors the winner of a match had against the loser (no aces/dfs)",
            labels = {
                "metric": "Metric",
                "margin": "Differential",
            },
            width=800,
            height=800,
            )

fig1_rally.update_layout(showlegend=False)

fig1_rally.show()

Once aces and double faults are removed, the median wniners margin drops from +7 to +5, and the general shape is less spread (IQR, range, outliers). In contrast, the median errors margin barely changes from +6 to +5, with the IQR only slightly shrinking (14 to -3 becoming 13 to -3.5). This is a meaningful finding: that the serve is an extremely large portion of playing winning tennis. Once the serve is removed, the two margins are quite close to identical, implying that both playstyles are at a pretty even playing field. 

We then check if this rally-only pattern still holds up by surface.

In [21]:
fig2_rally = px.box(df_long_rally, x="tournament_name", y="margin", color="metric",
            title="Clay (French Open) shows the most variation even with serves removed",
            subtitle="Side by side of how many more/less winners/unforced errors the winner of a match had against the loser (by tournament)",
            labels = {
                "metric": "",
                "margin": "Margin",
                "tournament_name": "Tournament",
            },
            width=1200,
            height=800,
            )

fig2_rally.update_layout(
    # move the legend from the side to the bottom
    legend=dict(
        orientation="h",  
        yanchor="top",
        y=-0.08, 
        xanchor="center",
        x=0.5
    )
)

fig2_rally.show()

Across all 712 matches, our final conclusion is that players who win typically have a tiny edge in extra winners hit (+7) over a reduction in errors (+6), but given there is so much overlap between the two distributions, we cannot decide that this edge is decisive on its own. 

Splitting by tournament shows that this edge is surface-dependent and behaves as we would expect: on the surface that rewards tenacity, clay, the "fewer errors" margin narrowly edges the winners margin, while on hard-courts and grass, the winners edge  holds a bit more solidly. 

Once the serve statistics are removed, we come across our most striking finding. The winners margin drops, and its spread becomes far more narrow, while the errors margin stays largely the same. This suggests that a good part of the "advantage of being aggressive" really boils down to who serves better, not who takes more risk in a rally. Thus we conclude that, both risk-averse and risk-taking styles can win matches, but the serve appears to matter more towards the outcome then either playing philosophy.

It is worth noting that this is still merely associative rather than causal. Aggression, rally tolerance, and serving are all just small parts of a sport with countless interacting factors, many of which drive win chances. 